# 04 — Sémantique : un graphe de connaissances des vols, interrogeable en SPARQL

```
ontologie/vols.ttl (schéma) ─┐
sem_aeroports ───────────────┤
sem_compagnies ──────────────┼─► rdflib ─► ontologie/graphe_vols.ttl ─► Fuseki :3030 (SPARQL)
sem_routes (top 50) ─────────┤                                        └─► repli : rdflib
sem_vols (500 vols) ─────────┘
```

Les tables gold répondent aux questions **prévues** (une question = une table). Le graphe sert aux questions
qui **traversent** plusieurs notions à la fois : *« au départ de quels hubs les retards sont-ils surtout
contrôlables ? »* relie un vol à son aéroport, l'aéroport à sa classe (hub ou non), le retard à ses causes et
chaque cause à sa catégorie. L'ontologie décrit ces notions une seule fois ; les requêtes s'appuient sur les
classes au lieu de listes codées en dur.

**Contenu du graphe**

| Élément | Source | Nombre |
|---|---|---|
| Schéma : classes, propriétés, les 5 causes | `ontologie/vols.ttl` | — |
| Tous les aéroports desservis, avec leur État | `sem_aeroports` | 348 |
| Toutes les compagnies | `sem_compagnies` | 15 |
| Les routes au plus fort `score_priorite` | `sem_routes` | 50 |
| Un échantillon de vols sur ces routes, avec leurs retards par cause | `sem_vols` | 500 |
| Liens `owl:sameAs` des hubs vers Wikidata | table ci-dessous | 30 |

Les tables `sem_*` sont préparées par `03_gold` (section 9) à partir du silver et copiées dans PostgreSQL : ce
notebook n'a besoin que de pandas et de rdflib, **sans session Spark**. Le graphe ne contient qu'un échantillon
de vols : il sert à **montrer** ce qu'apporte une ontologie, pas à remplacer les tables gold pour les chiffres
nationaux.

## 1. Bibliothèques et lecture des tables `sem_*`

`rdflib` (graphes RDF en Python) n'est pas dans l'image Jupyter : la cellule l'installe au premier lancement.
`requests` sert à parler à Fuseki (section 6).

In [ ]:
import subprocess
import sys
from decimal import Decimal
from pathlib import Path

sys.path.append("/home/jovyan/work")
from commun.donnees import lire

try:
    import rdflib
except ImportError:   # absent de l'image quay.io/jupyter/pyspark-notebook : installé une seule fois
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdflib"], check=True)
    import rdflib

import pandas as pd
import requests
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import OWL, RDF, RDFS, XSD

# Racine du dépôt : /home/jovyan/work dans le conteneur Jupyter, le dossier parent sinon
RACINE = Path("/home/jovyan/work") if Path("/home/jovyan/work").exists() else Path.cwd().parent
SCHEMA = RACINE / "ontologie" / "vols.ttl"
SORTIE = RACINE / "ontologie" / "graphe_vols.ttl"
print(f"rdflib {rdflib.__version__} — schéma lu dans {SCHEMA.relative_to(RACINE).as_posix()}")

def milliers(n):
    """1234567 → « 1 234 567 » (séparateur de milliers français)."""
    return f"{int(n):,}".replace(",", " ")

In [ ]:
aeroports = lire("sem_aeroports")
compagnies = lire("sem_compagnies")
routes = lire("sem_routes")
vols = lire("sem_vols")

for nom, table in [("sem_aeroports", aeroports), ("sem_compagnies", compagnies),
                   ("sem_routes", routes), ("sem_vols", vols)]:
    print(f"{nom:<15} {len(table):>4} lignes   colonnes : {', '.join(table.columns)}")

## 2. Le schéma (T-Box) : `ontologie/vols.ttl`

On charge d'abord l'ontologie dans un graphe vide : les instances viendront s'y ajouter. Le schéma est
lui-même fait de triplets, on peut donc l'interroger en SPARQL comme les données. Ci-dessous, les classes et
leur classe mère : on retrouve les deux hiérarchies du modèle, **Hub ⊂ Aeroport** et
**CauseControlable / CauseNonControlable ⊂ CauseRetard**.

Espaces de noms : `vols:` pour le vocabulaire ; un espace par type d'instance pour les données (`aer:ATL`,
`cie:DL`, `route:LAX-SFO`, `vol:20240101_B6_623_JFK_14`, `etat:GA`, `imp:` pour les imputations), et `wd:`
pour Wikidata.

In [ ]:
VOLS = Namespace("http://example.org/vols/ontologie#")
AER = Namespace("http://example.org/vols/aeroport/")
CIE = Namespace("http://example.org/vols/compagnie/")
ROUTE = Namespace("http://example.org/vols/route/")
VOL = Namespace("http://example.org/vols/vol/")
ETAT = Namespace("http://example.org/vols/etat/")
IMP = Namespace("http://example.org/vols/imputation/")
WD = Namespace("http://www.wikidata.org/entity/")
PREFIXES = {"vols": VOLS, "aer": AER, "cie": CIE, "route": ROUTE, "vol": VOL, "etat": ETAT, "imp": IMP,
            "wd": WD, "owl": OWL, "rdfs": RDFS, "xsd": XSD}

g = Graph()
g.parse(SCHEMA, format="turtle")
for prefixe, espace in PREFIXES.items():
    g.bind(prefixe, espace)
n_schema = len(g)
print(f"Schéma : {n_schema} triplets")

hierarchie = g.query("""
    SELECT ?classe ?libelle ?mere WHERE {
        ?classe a owl:Class ; rdfs:label ?libelle .
        FILTER (isIRI(?classe) && lang(?libelle) = "fr")
        OPTIONAL { ?classe rdfs:subClassOf ?mere }
    }
    ORDER BY ?mere ?classe
""", initNs=PREFIXES)
pd.DataFrame([(g.qname(c), str(l), g.qname(m) if m else "") for c, l, m in hierarchie],
             columns=["classe", "libellé", "sous-classe de"])

## 3. Les instances (A-Box)

**Conventions**
- Chaque valeur est écrite avec son type XSD (`xsd:decimal`, `xsd:integer`, `xsd:date`, `xsd:boolean`) : SPARQL
  peut alors comparer, sommer et trier sans conversion.
- **Une valeur manquante ne donne aucun triplet.** RDF n'a pas de `NULL` : un vol annulé n'a simplement pas de
  `vols:retardArriveeMin`. Les requêtes le traitent avec `OPTIONAL` ou `FILTER NOT EXISTS`.
- Les fonctions ci-dessous centralisent ces règles : `ajouter()` n'écrit le triplet que si la valeur existe.

In [ ]:
def vide(valeur):
    """Vrai pour une valeur manquante (None, NaN, NaT) : on n'écrit alors aucun triplet."""
    return valeur is None or (not isinstance(valeur, str) and pd.isna(valeur))


def decimal(valeur, chiffres=4):
    """Littéral xsd:decimal arrondi (format fixe, jamais de notation 1e-05)."""
    return Literal(format(Decimal(str(round(float(valeur), chiffres))), "f"), datatype=XSD.decimal)


def entier(valeur):
    return Literal(int(valeur))                      # xsd:integer


def texte(valeur):
    """Chaîne ; un nombre entier lu en flottant (623.0) redevient « 623 »."""
    if isinstance(valeur, float) and valeur.is_integer():
        valeur = int(valeur)
    return Literal(str(valeur))


def ajouter(sujet, propriete, valeur, convertir=texte):
    if not vide(valeur):
        g.add((sujet, propriete, convertir(valeur)))

### 3.1 États et aéroports

Les 348 aéroports desservis sont tous dans le graphe. Un aéroport qui assure au moins 1 % des départs
nationaux (`est_hub`, calculé dans `03_gold`) est typé **`vols:Hub`**, les autres **`vols:Aeroport`**.

On n'écrit **pas** `aer:ATL a vols:Aeroport` pour un hub : c'est une conséquence de `vols:Hub rdfs:subClassOf
vols:Aeroport`. Les requêtes retrouvent tous les aéroports, hubs compris, avec le chemin
`?a a/rdfs:subClassOf* vols:Aeroport` (« a pour type une classe qui est, directement ou non, une sous-classe
d'Aeroport »). C'est la hiérarchie qui fait le travail, pas une colonne en double.

Chaque aéroport est rattaché à son **État** (`vols:situeDans`). Le nom des États est donné en français ; `TT`
est le code BTS des territoires américains du Pacifique (Guam, Saipan).

In [ ]:
NOMS_ETATS = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "Californie", "CO": "Colorado",
    "CT": "Connecticut", "DE": "Delaware", "DC": "District de Columbia", "FL": "Floride", "GA": "Géorgie",
    "HI": "Hawaï", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiane", "ME": "Maine", "MD": "Maryland", "MA": "Massachusetts",
    "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri", "MT": "Montana",
    "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey", "NM": "Nouveau-Mexique",
    "NY": "New York", "NC": "Caroline du Nord", "ND": "Dakota du Nord", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvanie", "RI": "Rhode Island", "SC": "Caroline du Sud",
    "SD": "Dakota du Sud", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginie", "WA": "Washington", "WV": "Virginie-Occidentale", "WI": "Wisconsin", "WY": "Wyoming",
    "PR": "Porto Rico", "VI": "Îles Vierges des États-Unis", "GU": "Guam", "AS": "Samoa américaines",
    "MP": "Îles Mariannes du Nord", "TT": "Territoires du Pacifique (Guam, Saipan)",
}

etats = sorted(aeroports["etat"].dropna().unique())
for code in etats:
    g.add((ETAT[code], RDF.type, VOLS.Etat))
    g.add((ETAT[code], VOLS.codeEtat, Literal(code)))
    g.add((ETAT[code], RDFS.label, Literal(NOMS_ETATS.get(code, code), lang="fr")))
sans_nom = [c for c in etats if c not in NOMS_ETATS]
print(f"{len(etats)} États ou territoires" + (f" (sans nom connu : {sans_nom})" if sans_nom else ""))

for a in aeroports.itertuples(index=False):
    s = AER[a.code_iata]
    g.add((s, RDF.type, VOLS.Hub if a.est_hub else VOLS.Aeroport))
    g.add((s, VOLS.codeIATA, Literal(a.code_iata)))
    ajouter(s, RDFS.label, a.nom)
    ajouter(s, VOLS.ville, a.ville)
    ajouter(s, VOLS.latitude, a.lat, decimal)
    ajouter(s, VOLS.longitude, a.lon, decimal)
    ajouter(s, VOLS.departs, a.departs, entier)
    ajouter(s, VOLS.partDepartsPct, a.part_departs_pct, lambda v: decimal(v, 3))
    ajouter(s, VOLS.pageWikipedia, a.wikipedia, lambda url: Literal(url, datatype=XSD.anyURI))
    if not vide(a.etat):
        g.add((s, VOLS.situeDans, ETAT[a.etat]))

n_hubs = int(aeroports["est_hub"].sum())
print(f"{len(aeroports)} aéroports, dont {n_hubs} hubs")

### 3.2 Liens vers Wikidata (`owl:sameAs`)

`aer:ATL owl:sameAs wd:Q214861` dit que notre ressource « ATL » et l'élément Wikidata Q214861 désignent **le même
aéroport**. Le graphe sort ainsi de son silo : tout ce que Wikidata sait de l'aéroport (nom officiel dans toutes
les langues, nombre de passagers, compagnies qui y ont leur base, coordonnées) devient accessible en suivant le
lien, par exemple avec une requête fédérée (`SERVICE <https://query.wikidata.org/sparql>`).

**Comment les identifiants ont été obtenus** (le 26/11/2026, sur https://query.wikidata.org) : la requête
ci-dessous cherche les éléments dont le code IATA (propriété `P238`) est celui d'un hub et dont le pays (`P17`)
est les États-Unis (`Q30`).

```sparql
SELECT ?iata ?item ?itemLabel WHERE {
  VALUES ?iata { "ATL" "DFW" "DEN" … }
  ?item wdt:P238 ?iata ; wdt:P17 wd:Q30 .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
```

Deux codes avaient deux candidats, départagés à la main : **AUS** (Q1430913, l'aéroport actuel, et non Q7347861,
l'ancien aéroport Robert Mueller, fermé en 1999) et **EWR** (Q466786, l'aéroport, et non Q14705806, sa gare).
Les identifiants sont recopiés ici plutôt que redemandés à chaque exécution : le notebook reste reproductible
sans accès à Internet, et le choix des deux homonymes est vérifiable.

In [ ]:
WIKIDATA_HUBS = {
    "ATL": "Q214861",  "AUS": "Q1430913", "BNA": "Q1432146", "BOS": "Q671475",  "BWI": "Q94895",
    "CLT": "Q597649",  "DAL": "Q613569",  "DCA": "Q676576",  "DEN": "Q330015",  "DFW": "Q459096",
    "DTW": "Q652505",  "EWR": "Q466786",  "FLL": "Q635361",  "IAH": "Q744748",  "JFK": "Q8685",
    "LAS": "Q853886",  "LAX": "Q8731",    "LGA": "Q319654",  "MCO": "Q929859",  "MDW": "Q1061437",
    "MIA": "Q127530",  "MSP": "Q875749",  "ORD": "Q213717",  "PHL": "Q512128",  "PHX": "Q912916",
    "SAN": "Q1333923", "SEA": "Q14295",   "SFO": "Q8688",    "SLC": "Q526488",  "TPA": "Q571536",
}

hubs = aeroports.loc[aeroports["est_hub"], "code_iata"].tolist()
for code in hubs:
    if code in WIKIDATA_HUBS:
        g.add((AER[code], OWL.sameAs, WD[WIKIDATA_HUBS[code]]))

sans_lien = sorted(set(hubs) - set(WIKIDATA_HUBS))
print(f"{len(hubs) - len(sans_lien)} hubs sur {len(hubs)} reliés à Wikidata"
      + (f" ; à compléter : {', '.join(sans_lien)}" if sans_lien else ""))

### 3.3 Compagnies et routes prioritaires

Les 15 compagnies sont toutes dans le graphe. Les **50 routes** sont celles au plus fort `score_priorite`
(≈ nombre de vols en retard sur l'année) : ce sont les routes que le dashboard recommande de traiter en premier.
Chaque route garde ses indicateurs gold et son **rang de priorité** (1 = la plus prioritaire), et pointe vers ses
aéroports d'origine et de destination.

In [ ]:
for c in compagnies.itertuples(index=False):
    s = CIE[c.compagnie_code]
    g.add((s, RDF.type, VOLS.Compagnie))
    g.add((s, VOLS.codeCompagnie, Literal(c.compagnie_code)))
    ajouter(s, RDFS.label, c.compagnie_nom)

routes = routes.sort_values(["score_priorite", "origine", "destination"],
                            ascending=[False, True, True]).reset_index(drop=True)
for rang, r in enumerate(routes.itertuples(index=False), start=1):
    s = ROUTE[f"{r.origine}-{r.destination}"]
    g.add((s, RDF.type, VOLS.Route))
    g.add((s, RDFS.label, Literal(f"{r.origine} → {r.destination}")))
    g.add((s, VOLS.origine, AER[r.origine]))
    g.add((s, VOLS.destination, AER[r.destination]))
    g.add((s, VOLS.rangPriorite, Literal(rang)))
    ajouter(s, VOLS.nbVols, r.vols, entier)
    ajouter(s, VOLS.pctALHeure, r.pct_a_l_heure, lambda v: decimal(v, 2))
    ajouter(s, VOLS.retardMoyenMin, r.retard_moyen_min, lambda v: decimal(v, 1))
    ajouter(s, VOLS.scorePriorite, r.score_priorite, entier)

print(f"{len(compagnies)} compagnies, {len(routes)} routes : de {routes.origine[0]} → {routes.destination[0]} "
      f"(score {milliers(routes.score_priorite[0])}) à {routes.origine.iloc[-1]} → {routes.destination.iloc[-1]} "
      f"(score {milliers(routes.score_priorite.iloc[-1])})")

### 3.4 Vols et imputation des retards par cause

Pour chaque vol de l'échantillon : sa compagnie, ses aéroports, sa route, sa date, ses retards et ses
indicateurs. `vols:estEnRetard` (retard ≥ 15 min, définition du DOT) n'est écrit que pour un vol **opéré** : un vol
annulé ou détourné n'entre pas dans la ponctualité, comme dans le gold.

**Imputations.** Pour un vol arrivé en retard, la BTS répartit les minutes entre cinq causes. Chaque cause non
nulle devient un nœud `vols:ImputationRetard` (relation n-aire : vol × cause × minutes) :

```
vol:V  vols:aImputation  imp:V_meteo .
imp:V_meteo  a vols:ImputationRetard ;  vols:cause vols:cause_meteo ;  vols:minutes 23.0 .
```

**Annulations.** Le code d'annulation BTS (A compagnie, B météo, C NAS, D sécurité) pointe vers **les mêmes
causes** que les retards (`vols:causeAnnulation`) : une annulation se range donc elle aussi dans la hiérarchie
contrôlable / non contrôlable.

In [ ]:
CAUSES = {   # code de la cause (gold_causes_retard.cause) → colonne de sem_vols
    "compagnie": "retard_compagnie_min",
    "avion_retard": "retard_avion_min",
    "meteo": "retard_meteo_min",
    "nas": "retard_nas_min",
    "securite": "retard_securite_min",
}
ANNULATION = {"A": "compagnie", "B": "meteo", "C": "nas", "D": "securite"}   # CancellationCode BTS

n_imputations = 0
for v in vols.itertuples(index=False):
    s = VOL[v.id_vol]
    date = pd.Timestamp(v.date_vol).date()
    g.add((s, RDF.type, VOLS.Vol))
    g.add((s, VOLS.idVol, Literal(v.id_vol)))
    g.add((s, RDFS.label, Literal(f"{v.compagnie_code} {texte(v.numero_vol)} {v.origine} → {v.destination} "
                                  f"du {date:%d/%m/%Y}")))
    g.add((s, VOLS.dateVol, Literal(date)))
    ajouter(s, VOLS.numeroVol, v.numero_vol)
    g.add((s, VOLS.operePar, CIE[v.compagnie_code]))
    g.add((s, VOLS.aeroportDepart, AER[v.origine]))
    g.add((s, VOLS.aeroportArrivee, AER[v.destination]))
    g.add((s, VOLS.surRoute, ROUTE[f"{v.origine}-{v.destination}"]))
    ajouter(s, VOLS.heureDepartPrevue, v.heure_depart_prevue, entier)
    ajouter(s, VOLS.distanceMiles, v.distance_miles, lambda x: decimal(x, 1))
    ajouter(s, VOLS.retardDepartMin, v.retard_depart_min, lambda x: decimal(x, 1))
    ajouter(s, VOLS.retardArriveeMin, v.retard_arrivee_min, lambda x: decimal(x, 1))
    g.add((s, VOLS.estAnnule, Literal(bool(v.annule))))
    g.add((s, VOLS.estDetourne, Literal(bool(v.detourne))))

    opere = not v.annule and not v.detourne
    if opere and not vide(v.retard_15):
        g.add((s, VOLS.estEnRetard, Literal(int(v.retard_15) == 1)))
    if v.annule and v.code_annulation in ANNULATION:
        g.add((s, VOLS.causeAnnulation, VOLS[f"cause_{ANNULATION[v.code_annulation]}"]))

    for cause, colonne in CAUSES.items():
        minutes = getattr(v, colonne)
        if not vide(minutes) and float(minutes) > 0:
            i = IMP[f"{v.id_vol}_{cause}"]
            g.add((s, VOLS.aImputation, i))
            g.add((i, RDF.type, VOLS.ImputationRetard))
            g.add((i, VOLS.cause, VOLS[f"cause_{cause}"]))
            g.add((i, VOLS.minutes, decimal(minutes, 1)))
            n_imputations += 1

print(f"{len(vols)} vols, {n_imputations} imputations de retard — graphe : {milliers(len(g))} triplets "
      f"(dont {n_schema} pour le schéma)")

Un vol en retard de l'échantillon, tel qu'il est écrit dans le graphe (le vol et ses imputations) :

In [ ]:
exemple = next(iter(g.query("""
    SELECT ?vol WHERE { ?vol vols:estEnRetard true ; vols:aImputation ?i . }
    GROUP BY ?vol HAVING (COUNT(?i) >= 2) ORDER BY ?vol LIMIT 1
""", initNs=PREFIXES)))[0]

extrait = Graph(namespace_manager=g.namespace_manager)
for p, o in g.predicate_objects(exemple):
    extrait.add((exemple, p, o))
    if p == VOLS.aImputation:
        for p2, o2 in g.predicate_objects(o):
            extrait.add((o, p2, o2))
print(extrait.serialize(format="turtle"))

## 4. Contrôles de cohérence du graphe

Avant d'exporter, on vérifie en SPARQL que le graphe redit bien les tables :
1. **nombre d'instances par classe** : 15 compagnies, 50 routes, 500 vols, aéroports et hubs ;
2. **la hiérarchie fonctionne** : le chemin `a/rdfs:subClassOf* vols:Aeroport` retrouve les 348 aéroports, hubs
   compris, alors qu'aucun hub n'est typé `vols:Aeroport` explicitement ;
3. **les imputations redonnent le retard** : pour chaque vol en retard, la somme des minutes par cause est égale
   au retard à l'arrivée (c'est la règle R8 du silver) ;
4. **chaque vol est sur la bonne route** : l'origine de sa route est son aéroport de départ.

In [ ]:
def requete_locale(sparql):
    """Exécute une requête SELECT sur le graphe rdflib et renvoie un DataFrame (IRI abrégées en préfixe:nom)."""
    resultat = g.query(sparql, initNs=PREFIXES)
    colonnes = [str(v) for v in resultat.vars]
    lignes = []
    for ligne in resultat:
        valeurs = []
        for x in ligne:
            if isinstance(x, rdflib.URIRef):
                valeurs.append(g.namespace_manager.normalizeUri(x).strip("<>"))
            elif isinstance(x, rdflib.Literal):
                py = x.toPython()
                valeurs.append(float(py) if isinstance(py, Decimal) else py)
            else:
                valeurs.append(x)
        lignes.append(valeurs)
    return pd.DataFrame(lignes, columns=colonnes)


par_classe = requete_locale("""
    SELECT ?classe (COUNT(?x) AS ?instances) WHERE {
        ?x a ?classe .
        FILTER (STRSTARTS(STR(?classe), STR(vols:)))
    }
    GROUP BY ?classe ORDER BY DESC(?instances)
""")
display(par_classe)

tous_aeroports = requete_locale("SELECT (COUNT(DISTINCT ?a) AS ?n) WHERE { ?a a/rdfs:subClassOf* vols:Aeroport }")
incoherents = requete_locale("""
    SELECT (COUNT(*) AS ?n) WHERE {
        { SELECT ?vol (SUM(?m) AS ?total) WHERE { ?vol vols:aImputation/vols:minutes ?m } GROUP BY ?vol }
        ?vol vols:estEnRetard true ; vols:retardArriveeMin ?retard .
        FILTER (ABS(?total - ?retard) > 0.5)
    }
""")
mauvaise_route = requete_locale("""
    SELECT (COUNT(?vol) AS ?n) WHERE {
        ?vol vols:surRoute ?route ; vols:aeroportDepart ?depart .
        FILTER NOT EXISTS { ?route vols:origine ?depart }
    }
""")

controles = {
    "aéroports retrouvés par la hiérarchie": (int(tous_aeroports.n[0]), len(aeroports)),
    "vols en retard dont les causes ≠ le retard": (int(incoherents.n[0]), 0),
    "vols dont la route ne part pas de leur aéroport": (int(mauvaise_route.n[0]), 0),
}
for nom, (obtenu, attendu) in controles.items():
    print(f"{'✓' if obtenu == attendu else '✗'} {nom} : {obtenu} (attendu : {attendu})")
assert all(obtenu == attendu for obtenu, attendu in controles.values())

## 5. Export : `ontologie/graphe_vols.ttl`

Le fichier contient **le schéma et les instances** : il se suffit à lui-même et se charge en une fois dans
Fuseki (ou dans n'importe quel outil RDF, comme Protégé). Il est versionné, contrairement aux données brutes : il
est petit et c'est un livrable du projet.

In [ ]:
g.serialize(SORTIE, format="turtle", encoding="utf-8")
print(f"{SORTIE.relative_to(RACINE).as_posix()} : {milliers(len(g))} triplets, "
      f"{milliers(SORTIE.stat().st_size / 1024)} Ko")

# Relecture : le fichier exporté redonne exactement le même graphe
relu = Graph().parse(SORTIE, format="turtle")
assert len(relu) == len(g), (len(relu), len(g))
print("Relecture OK")